# Train the transformer on Kaggle

Builds `wav-transformer` from GitHub and trains the RM (or Vec) transformer on XQuAD (de). The trained model ends up
in the **Output** of this notebook, ready to download and to continue in the next session.

**Once**
1. Create a Kaggle account and verify your phone number (*Settings → Phone verification*); without it notebooks have no internet.
2. *Create → New Notebook*, then *File → Import Notebook* and upload this file.
3. In the right panel (*Session options*): **Internet on**, Accelerator **None** (CPU) with `USE_GPU = False`: at model width 32
   the GPU is not faster than the CPU (it copies every product to the GPU and back) and its weekly quota is saved.
   For a GPU set Accelerator **GPU T4 x2** and `USE_GPU = True`.

**Every run**
1. Adjust the settings in the first code cell.
2. *Save Version → Save & Run All (Commit)*. The run continues in the background (you can close the browser) for up to 12 hours;
   `MAX_HOURS` stops the training before that and keeps the best model.
3. When it is done: open the version → **Output** → download `model_rm.zip` (or use it as input for the next run). `loss_curve.png` there shows training and validation loss per epoch, over all continued runs (`loss_history.csv`).

**Faster start**: the output also holds the built binary (`bin/`). With an earlier output as input (*Add Input*), the next
run reuses it and skips the ~12 minute build, unless the code (`src/`, `Cargo.toml`, ...) changed since or `USE_GPU` differs.

**Continue in the next session**
*Add Input → Your Work → Notebooks* → choose this notebook (its last output). With `RESUME = True` the training continues that model.

Models of different width (e.g. 16 or 128 before, 32 now) cannot be continued: start with `RESUME = False` or without input.

**Use the model on your computer**: unpack `model_rm.zip` into the `STORAGE` folder of the project (it contains the model file
`transformer_rm_v1` and `transformer_rm_v1.split.json`).

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
MODEL = "rm"            # "rm" (RM transformer) or "vec" (Vec transformer)
EPOCHS = 40
BATCH_SIZE = 16
LEARNING_RATE = 0.0003
RESUME = True           # continue the model found in the notebook input; False starts a new model
USE_GPU = False         # True: build with --features cuda (needs Accelerator GPU); not faster at width 32
MAX_HOURS = 11.0        # Kaggle ends a session after 12 h: stop before, the best model stays saved

INPUT_ROOT = "/kaggle/input"      # previous outputs added with "Add Input"
OUTPUT_ROOT = "/kaggle/working"   # everything here becomes the notebook output
SOURCE_DIR = "/tmp/rustCrates"    # checkout and build outside the output (the build is several GB)

In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

MODEL_DIR = {"rm": "transformer_rm_v1", "vec": "transformer_01.04.2025"}[MODEL]
COMMAND = {"rm": "train", "vec": "train-vec"}[MODEL]

def copy_model(source, target):
    # The model is a file (older builds: a sled folder); copy either.
    if os.path.isdir(source):
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(source, target)
    else:
        shutil.copy(source, target)

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

# Rust (the repository's rust-toolchain file selects the version on the first cargo call)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Source
if os.path.exists(SOURCE_DIR):
    run(f"git fetch --depth 200 origin {BRANCH} && git checkout -B {BRANCH} FETCH_HEAD", cwd=SOURCE_DIR)
else:
    run(f"git clone --depth 200 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)

In [ ]:
def find_previous_model(input_root, model_dir):
    # The model from an earlier output, or a model zip to unpack. Returns the folder that contains model_dir.
    for path in sorted(glob.glob(os.path.join(input_root, "**", model_dir), recursive=True)):
        return os.path.dirname(path)
    for archive in sorted(glob.glob(os.path.join(input_root, "**", f"model_{MODEL}.zip"), recursive=True)):
        target = "/tmp/previous_model"
        shutil.rmtree(target, ignore_errors=True)
        with zipfile.ZipFile(archive) as z:
            z.extractall(target)
        if os.path.exists(os.path.join(target, model_dir)):
            return target
    return None

storage = os.path.join(SOURCE_DIR, "STORAGE")
os.makedirs(storage, exist_ok=True)
previous = find_previous_model(INPUT_ROOT, MODEL_DIR) if RESUME else None
if previous:
    copy_model(os.path.join(previous, MODEL_DIR), os.path.join(storage, MODEL_DIR))
    split = os.path.join(previous, MODEL_DIR + ".split.json")
    if os.path.exists(split):
        shutil.copy(split, storage)
    print("Continuing the model from", previous)
else:
    print("Training a new model" + (" (no previous model in the input)" if RESUME else ""))
START = "resume" if previous else "new"

In [ ]:
# Build (~12 minutes), or reuse the binary of an earlier output (Add Input) when the code is unchanged
features = "--features cuda" if USE_GPU else ""
binary = os.path.join(os.environ.get("CARGO_TARGET_DIR", os.path.join(SOURCE_DIR, "target")), "release", "wav-transformer")
commit = subprocess.run("git rev-parse HEAD", shell=True, cwd=SOURCE_DIR, capture_output=True, text=True).stdout.strip()

# Files whose change needs a new binary; data (datasets/), docs and the notebook do not.
BUILD_INPUTS = "src Cargo.toml Cargo.lock build.rs rust-toolchain"

def reusable_binary():
    # (path, build info) of a cached binary built from code identical to this checkout with the same features.
    for info_path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "bin", "build.json"), recursive=True)):
        with open(info_path) as f:
            info = json.load(f)
        cached = os.path.join(os.path.dirname(info_path), "wav-transformer")
        if info.get("features") != features or not os.path.exists(cached):
            continue
        if info.get("commit") == commit:
            return cached, info
        # Same code if nothing that goes into the binary changed between the two commits (needs that commit in history).
        diff = subprocess.run(f"git diff --quiet {info.get('commit')} HEAD -- {BUILD_INPUTS}", shell=True, cwd=SOURCE_DIR)
        if diff.returncode == 0:
            return cached, info
        print(f"Cached binary of {info.get('commit', '?')[:7]} is outdated (code changed, or that commit is not in the history)")
    return None, None

cached, info = reusable_binary()
if cached:
    os.makedirs(os.path.dirname(binary), exist_ok=True)
    shutil.copy(cached, binary)
    os.chmod(binary, 0o755)
    print(f"Using the cached binary built from {info['commit'][:7]} ({cached}); no build needed")
else:
    run(f"cargo build --release {features}", cwd=SOURCE_DIR)
    info = {"commit": commit, "features": features}

# Keep the binary in the output, so the next run (with this output as input) can skip the build.
bin_dir = os.path.join(OUTPUT_ROOT, "bin")
os.makedirs(bin_dir, exist_ok=True)
shutil.copy(binary, os.path.join(bin_dir, "wav-transformer"))
with open(os.path.join(bin_dir, "build.json"), "w") as f:
    json.dump(info, f)


In [ ]:
# Train; the full log is kept in the output as train.log
env = dict(os.environ, TRAIN_MAX_HOURS=str(MAX_HOURS))
cmd = [binary, COMMAND, str(EPOCHS), "all", str(BATCH_SIZE), START, str(LEARNING_RATE)]
print("$", " ".join(cmd), flush=True)
started = time.time()
with open(os.path.join(OUTPUT_ROOT, "train.log"), "w") as log:
    proc = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        if any(key in line for key in ("Loaded", "Created", "Epoch", "LOSS", "best", "improvement", "Learning rate", "selected", "stop", "Stop", "error", "Error", "panicked", "done")):
            print(f"[{(time.time() - started) / 3600:5.2f} h] {line.rstrip()}", flush=True)
    proc.wait()
print("exit code", proc.returncode)

In [ ]:
# Loss curve: loss_history.csv (all runs, continued runs appended) and loss_curve.png in the output
import csv, re
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def parse_log(path):
    """{epoch: {"train": loss, "val": loss}} from a train.log."""
    ansi = re.compile(r"\x1b\[[0-9;]*m")
    epochs = {}
    for line in open(path, encoding="utf-8", errors="replace"):
        m = re.search(r"Epoch: (\d+), (TRAINING|VALIDATION) LOSS: ([0-9.eE+-]+)", ansi.sub("", line))
        if m:
            epochs.setdefault(int(m.group(1)), {})["train" if m.group(2) == "TRAINING" else "val"] = float(m.group(3))
        # Older binaries printed the losses only every 5th epoch; an improvement line still holds the validation loss.
        m = re.search(r"New best validation loss: ([0-9.eE+-]+) at epoch (\d+)", ansi.sub("", line))
        if m:
            epochs.setdefault(int(m.group(2)), {}).setdefault("val", float(m.group(1)))
    return epochs

# Earlier runs of a continued model (only with RESUME, otherwise the curve starts fresh)
rows = []
if RESUME:
    previous = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "loss_history.csv"), recursive=True))
    if previous:
        with open(previous[0]) as f:
            rows = [{"epoch": int(r["epoch"]), "run": int(r["run"]), "train": r["train"], "val": r["val"]} for r in csv.DictReader(f)]
        # The train.log next to it may hold more of its last run than the csv (older notebook versions): re-read it.
        log = os.path.join(os.path.dirname(previous[0]), "train.log")
        last = max((r["run"] for r in rows), default=0)
        if last and os.path.exists(log):
            parsed = parse_log(log)
            if len(parsed) > sum(r["run"] == last for r in rows):
                start = min(r["epoch"] for r in rows if r["run"] == last)
                rows = [r for r in rows if r["run"] != last] + [
                    {"epoch": start + e, "run": last, "train": l.get("train", ""), "val": l.get("val", "")} for e, l in sorted(parsed.items())]
        print("Earlier history:", previous[0], len(rows), "epochs")
    else:
        # An output from before this cell existed: its train.log
        logs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "train.log"), recursive=True))
        if logs:
            rows = [{"epoch": e, "run": 1, "train": l.get("train", ""), "val": l.get("val", "")} for e, l in sorted(parse_log(logs[0]).items())]
            print("Earlier history:", logs[0], len(rows), "epochs")
run = max((r["run"] for r in rows), default=0) + 1
offset = max((r["epoch"] for r in rows), default=-1) + 1
for epoch, losses in sorted(parse_log(os.path.join(OUTPUT_ROOT, "train.log")).items()):
    rows.append({"epoch": offset + epoch, "run": run, "train": losses.get("train", ""), "val": losses.get("val", "")})

with open(os.path.join(OUTPUT_ROOT, "loss_history.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "run", "train", "val"])
    writer.writeheader()
    writer.writerows(rows)

def series(key):
    points = [(r["epoch"], float(r[key])) for r in rows if r[key] != ""]
    return [p[0] for p in points], [p[1] for p in points]

if rows:
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)
    ax.plot(*series("train"), label="Training", color="#2a6fdb", linewidth=2)
    val_x, val_y = series("val")
    ax.plot(val_x, val_y, label="Validation", color="#e07b00", linewidth=2)
    if val_y:
        best = min(range(len(val_y)), key=val_y.__getitem__)
        ax.scatter([val_x[best]], [val_y[best]], color="#e07b00", zorder=3)
        ax.annotate(f"best {val_y[best]:.3f} (epoch {val_x[best]})", (val_x[best], val_y[best]), textcoords="offset points", xytext=(0, 10), ha="center")
    for before, r in zip(rows, rows[1:]):
        if r["run"] != before["run"]:
            ax.axvline(r["epoch"] - 0.5, color="#999999", linestyle=":", linewidth=1)
    ax.set_xlabel("Epoch (dotted line: new Kaggle run)")
    ax.set_ylabel("Loss per token")
    ax.set_title(f"{MODEL_DIR}: loss per epoch")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_ROOT, "loss_curve.png"))
    plt.show()
    print("Saved loss_curve.png and loss_history.csv:", len(rows), "epochs")
else:
    print("No epoch finished: no loss curve")


In [ ]:
# Output: the model, its training/validation split and one zip of both
output = os.path.join(OUTPUT_ROOT, "model")
shutil.rmtree(output, ignore_errors=True)
os.makedirs(output)
copy_model(os.path.join(storage, MODEL_DIR), os.path.join(output, MODEL_DIR))
split = os.path.join(storage, MODEL_DIR + ".split.json")
if os.path.exists(split):
    shutil.copy(split, output)
archive = shutil.make_archive(os.path.join(OUTPUT_ROOT, f"model_{MODEL}"), "zip", output)
print("Saved", archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")